In [63]:
import numpy as np
import scipy.io.wavfile as wav
import scipy.signal as signal
import matplotlib.pyplot as plt
import librosa
import os
from glob import glob

In [64]:
music_file = 'music_decomposition/FoxTitle.wav'

In [65]:
sample_rate, music_signal = wav.read(music_file)

/var/folders/yj/v8_dnf_j2cv3dhbnc108tr7w0000gn/T/ipykernel_63385/2373438420.py:1: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sample_rate, music_signal = wav.read(music_file)


In [66]:
window_size = 1024
hop_size = 160
window = signal.windows.hann(window_size)


In [67]:
# STFT for the full music signal
frequencies, times, Zxx = signal.stft(
    music_signal,
    fs=sample_rate,
    window=window,
    nperseg=window_size,
    noverlap=window_size - hop_size,
    nfft=window_size,
    detrend=False,
    return_onesided=True,
    boundary=None,
    padded=False,
    axis=-1
)


In [68]:
M = np.abs(Zxx)
#M = 20 * np.log10(M + 1e-10)

In [69]:
U, S, Vt = np.linalg.svd(M, full_matrices=False)

In [70]:
num_bases = 11
U_bases = U[:, :num_bases]


In [71]:
np.savetxt('musicbases.csv', U_bases, delimiter=',')

In [72]:
notes_dir = 'music_decomposition/notes/'

In [73]:
note_files = glob(os.path.join(notes_dir, '*.wav'))

In [79]:
print(f"Note files: {note_files}")


Note files: []


In [74]:
ground_truth_spectra = {}

In [75]:
for note_file in note_files:
    # Extract the note name from the filename
    note_name = os.path.splitext(os.path.basename(note_file))[0]

    # Load the note audio file
    sr, note_signal = wav.read(note_file)

    # Compute STFT of the note
    f_note, t_note, Zxx_note = signal.stft(
        note_signal,
        fs=sr,
        window=window,
        nperseg=window_size,
        noverlap=window_size - hop_size,
        nfft=window_size,
        detrend=False,
        return_onesided=True,
        boundary=None,
        padded=False,
        axis=-1
    )

    # Use magnitude spectrogram
    M_note = np.abs(Zxx_note)

    # Consider frames 21 to 40 (steady part)
    start_frame = 20  # Python is zero-indexed
    end_frame = 40
    steady_frames = M_note[:, start_frame:end_frame]

    # Compute the average spectrum over these frames
    avg_spectrum = np.mean(steady_frames, axis=1)

    # Normalize the spectrum
    avg_spectrum /= np.linalg.norm(avg_spectrum) + 1e-10

    # Store in the dictionary
    ground_truth_spectra[note_name] = avg_spectrum


In [76]:
results = []

for i in range(num_bases):
    base = U_bases[:, i]

    # Normalize the base
    base_norm = np.linalg.norm(base)
    print(f"Base {i+1} norm: {base_norm}")
    if base_norm == 0:
        print(f"Base {i+1} has zero norm, skipping.")
        continue
    
    base_normalized = base / (base_norm + 1e-10)

    best_match = None
    highest_inner_product = -1

    for note_name, gt_spectrum in ground_truth_spectra.items():
        # Normalize the ground truth spectrum
        gt_norm = np.linalg.norm(gt_spectrum)
        print(f"Ground truth note {note_name} norm: {gt_norm}")
        if gt_norm == 0:
            print(f"Ground truth note {note_name} has zero norm, skipping.")
            continue
        
        gt_spectrum_normalized = gt_spectrum / (gt_norm + 1e-10)

        # Compute the inner product
        inner_product = np.dot(base_normalized, gt_spectrum_normalized)

        print(f"Base {i+1} vs {note_name}: Inner Product = {inner_product:.4f}")

        if inner_product > highest_inner_product:
            highest_inner_product = inner_product
            best_match = note_name

    if best_match is not None:
        results.append({
            'Base Index': i + 1,
            'Best Match Note': best_match,
            'Inner Product': highest_inner_product
        })
    else:
        results.append({
            'Base Index': i + 1,
            'Best Match Note': None,
            'Inner Product': -1
        })


Base 1 norm: 1.0
Base 2 norm: 0.9999999403953552
Base 3 norm: 1.0
Base 4 norm: 0.9999999403953552
Base 5 norm: 1.0
Base 6 norm: 1.0
Base 7 norm: 1.0
Base 8 norm: 1.0
Base 9 norm: 1.0
Base 10 norm: 0.9999999403953552
Base 11 norm: 1.0


In [77]:
print("Matching Results:")
unique_notes = set()

for result in results:
    print(f"Base {result['Base Index']}:")
    print(f"  Best Match Note: {result['Best Match Note']}")
    print(f"  Inner Product: {result['Inner Product']:.4f}\n")
    unique_notes.add(result['Best Match Note'])

print(f"Number of unique notes restored: {len(unique_notes)} out of 11")

# Optional: Plotting the bases and their matching ground truth spectra
for i, result in enumerate(results):
    base_index = result['Base Index'] - 1
    best_note = result['Best Match Note']

    # Check if a valid match was found
    if best_note is not None:
        plt.figure(figsize=(12, 5))

        # Plot the base spectrum
        plt.subplot(1, 2, 1)
        plt.plot(U_bases[:, base_index])
        plt.title(f"Base {base_index + 1} Spectrum")

        # Plot the ground truth spectrum
        plt.subplot(1, 2, 2)
        plt.plot(ground_truth_spectra[best_note])
        plt.title(f"Ground Truth Spectrum: {best_note}")

        plt.tight_layout()
        plt.show()
    else:
        print(f"No valid match found for Base {i + 1}")


Matching Results:
Base 1:
  Best Match Note: None
  Inner Product: -1.0000

Base 2:
  Best Match Note: None
  Inner Product: -1.0000

Base 3:
  Best Match Note: None
  Inner Product: -1.0000

Base 4:
  Best Match Note: None
  Inner Product: -1.0000

Base 5:
  Best Match Note: None
  Inner Product: -1.0000

Base 6:
  Best Match Note: None
  Inner Product: -1.0000

Base 7:
  Best Match Note: None
  Inner Product: -1.0000

Base 8:
  Best Match Note: None
  Inner Product: -1.0000

Base 9:
  Best Match Note: None
  Inner Product: -1.0000

Base 10:
  Best Match Note: None
  Inner Product: -1.0000

Base 11:
  Best Match Note: None
  Inner Product: -1.0000

Number of unique notes restored: 1 out of 11
No valid match found for Base 1
No valid match found for Base 2
No valid match found for Base 3
No valid match found for Base 4
No valid match found for Base 5
No valid match found for Base 6
No valid match found for Base 7
No valid match found for Base 8
No valid match found for Base 9
No valid 

In [78]:
for note_name, gt_spectrum in ground_truth_spectra.items():
    gt_norm = np.linalg.norm(gt_spectrum)
    print(f"Ground truth note {note_name} norm: {gt_norm}")
    if gt_norm == 0:
        print(f"Ground truth note {note_name} has zero norm, skipping.")
        continue
    
    gt_spectrum_normalized = gt_spectrum / (gt_norm + 1e-10)
